In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (they exist in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper to convert min/max timestamps to Asia/Shanghai timezone strings
def ts_range_info(ts_series):
    if ts_series.empty:
        return (None, None)
    tmin = pd.to_datetime(ts_series.min(), unit='s', utc=True).tz_convert(tz)
    tmax = pd.to_datetime(ts_series.max(), unit='s', utc=True).tz_convert(tz)
    # compact string representation with offset
    return (tmin.strftime('%Y-%m-%d %H:%M:%S %z'), tmax.strftime('%Y-%m-%d %H:%M:%S %z'))

# Summarize function for metric/trace/log files
def summarize_file(df, name_col=None, sample_limit_names=50, sample_limit_cmdb=20):
    row_count = df.shape[0]
    min_ts, max_ts = ts_range_info(df['timestamp']) if 'timestamp' in df.columns else (None, None)
    cmdb_unique = df['cmdb_id'].dropna().unique()
    cmdb_sample = list(cmdb_unique[:sample_limit_cmdb])
    cmdb_count = len(cmdb_unique)
    if name_col and name_col in df.columns:
        names_unique = df[name_col].dropna().unique()
        names_sample = list(names_unique[:sample_limit_names])
        names_count = len(names_unique)
    else:
        names_sample = []
        names_count = 0
    return {
        'rows': row_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'distinct_cmdb_count': cmdb_count,
        'distinct_cmdb_sample': cmdb_sample,
        'distinct_name_count': names_count,
        'distinct_name_sample': names_sample
    }

# Summarize metric, trace, log
metric_summary = summarize_file(metric_df, name_col='kpi_name', sample_limit_names=50)
trace_summary  = summarize_file(trace_df,  name_col='trace_name', sample_limit_names=50)
log_summary    = summarize_file(log_df,    name_col='log_name', sample_limit_names=50)

# Summarize error_logs (messages instead of KPI names), sample up to 20
def summarize_error_logs(df, sample_limit_msgs=20, sample_limit_cmdb=20):
    row_count = df.shape[0]
    min_ts, max_ts = ts_range_info(df['timestamp']) if 'timestamp' in df.columns else (None, None)
    cmdb_unique = df['cmdb_id'].dropna().unique()
    cmdb_sample = list(cmdb_unique[:sample_limit_cmdb])
    cmdb_count = len(cmdb_unique)
    if 'message' in df.columns:
        msgs_unique = df['message'].dropna().unique()
        msgs_sample = list(msgs_unique[:sample_limit_msgs])
        msgs_count = len(msgs_unique)
    else:
        msgs_sample = []
        msgs_count = 0
    return {
        'rows': row_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'distinct_cmdb_count': cmdb_count,
        'distinct_cmdb_sample': cmdb_sample,
        'distinct_message_count': msgs_count,
        'distinct_message_sample': msgs_sample
    }

error_logs_summary = summarize_error_logs(error_logs_df, sample_limit_msgs=20)

# Build a compact pandas DataFrame summarizing the four files
summary_df = pd.DataFrame([
    {
        'file': 'metric.csv',
        'rows': metric_summary['rows'],
        'min_timestamp': metric_summary['min_timestamp'],
        'max_timestamp': metric_summary['max_timestamp'],
        'distinct_cmdb_count': metric_summary['distinct_cmdb_count'],
        'distinct_cmdb_sample': metric_summary['distinct_cmdb_sample'],
        'distinct_kpi_count': metric_summary['distinct_name_count'],
        'distinct_kpi_sample_up_to_50': metric_summary['distinct_name_sample'][:50]
    },
    {
        'file': 'trace.csv',
        'rows': trace_summary['rows'],
        'min_timestamp': trace_summary['min_timestamp'],
        'max_timestamp': trace_summary['max_timestamp'],
        'distinct_cmdb_count': trace_summary['distinct_cmdb_count'],
        'distinct_cmdb_sample': trace_summary['distinct_cmdb_sample'],
        'distinct_trace_count': trace_summary['distinct_name_count'],
        'distinct_trace_sample_up_to_50': trace_summary['distinct_name_sample'][:50]
    },
    {
        'file': 'log.csv',
        'rows': log_summary['rows'],
        'min_timestamp': log_summary['min_timestamp'],
        'max_timestamp': log_summary['max_timestamp'],
        'distinct_cmdb_count': log_summary['distinct_cmdb_count'],
        'distinct_cmdb_sample': log_summary['distinct_cmdb_sample'],
        'distinct_log_count': log_summary['distinct_name_count'],
        'distinct_log_sample_up_to_50': log_summary['distinct_name_sample'][:50]
    },
    {
        'file': 'error_logs.csv',
        'rows': error_logs_summary['rows'],
        'min_timestamp': error_logs_summary['min_timestamp'],
        'max_timestamp': error_logs_summary['max_timestamp'],
        'distinct_cmdb_count': error_logs_summary['distinct_cmdb_count'],
        'distinct_cmdb_sample': error_logs_summary['distinct_cmdb_sample'],
        'distinct_message_count': error_logs_summary['distinct_message_count'],
        'distinct_message_sample_up_to_20': error_logs_summary['distinct_message_sample'][:20]
    }
])

# Keep output compact: show only the summary rows (4) and selected columns
summary_df = summary_df[[
    'file','rows','min_timestamp','max_timestamp',
    'distinct_cmdb_count','distinct_cmdb_sample',
    # the following columns vary per file; include all but they'll be NaN where not applicable
    'distinct_kpi_count','distinct_kpi_sample_up_to_50',
    'distinct_trace_count','distinct_trace_sample_up_to_50',
    'distinct_log_count','distinct_log_sample_up_to_50',
    'distinct_message_count','distinct_message_sample_up_to_20'
]]

summary_df

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files (all times UTC+8):\n\n"
    "metric.csv: 152,352 rows; time range 2022-03-21 21:30:00 +0800 → 2022-03-21 21:59:00 +0800; "
    "59 distinct cmdb_id (sample: adservice, adservice-0, adservice-1, ...); "
    "1,292 distinct kpi_name (sample: app.grpc.count, app.grpc.mrt, app.grpc.rr, ...).\n\n"
    "trace.csv: 31,536 rows; time range 2022-03-21 21:30:00 +0800 → 2022-03-21 21:59:00 +0800; "
    "40 distinct cmdb_id (sample: adservice-0, adservice-1, adservice-2, ...); "
    "200 distinct trace_name (sample: trace.from_frontend-0.duration_mean, ...).\n\n"
    "log.csv: 1,750 rows; time range 2022-03-21 21:30:00 +0800 → 2022-03-21 21:59:00 +0800; "
    "31 distinct cmdb_id (sample: adservice-0, adservice-1, adservice-2, cartservice-0, ...); "
    "2 distinct log_name: log.error_count, log.row_count.\n\n"
    "error_logs.csv: 743 rows; time range 2022-03-21 21:30:00 +0800 → 2022-03-21 21:59:55 +0800; "
    "3 distinct cmdb_id (frontend-1, frontend-0, frontend-2); 1 distinct message (example: "
    "'severity: warning, message: failed to retriev...')."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file    rows              min_timestamp              max_timestamp  distinct_cmdb_count                               distinct_cmdb_sample  distinct_kpi_count                       distinct_kpi_sample_up_to_50  distinct_trace_count                     distinct_trace_sample_up_to_50  distinct_log_count      distinct_log_sample_up_to_50  distinct_message_count                   distinct_message_sample_up_to_20
0      metric.csv  152352  2022-03-21 21:30:00 +0800  2022-03-21 21:59:00 +0800                   59  [adservice, adservice-0, adservice-1, adservic...              1292.0  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...                   NaN                                                NaN                 NaN                               NaN                     NaN                                                NaN
1       trace.csv   31536  2022-03-21 21:30:00 +0800  2022-03-21 21:59:00 +0800                   40  [adservice-0, adservice-1, adservice-2, adserv...                 NaN                                                NaN                 200.0  [trace.from_frontend-0.duration_mean, trace.fr...                 NaN                               NaN                     NaN                                                NaN
2         log.csv    1750  2022-03-21 21:30:00 +0800  2022-03-21 21:59:00 +0800                   31  [adservice-0, adservice-1, adservice-2, cartse...                 NaN                                                NaN                   NaN                                                NaN                 2.0  [log.error_count, log.row_count]                     NaN                                                NaN
3  error_logs.csv     743  2022-03-21 21:30:00 +0800  2022-03-21 21:59:55 +0800                    3               [frontend-1, frontend-0, frontend-2]                 NaN                                                NaN                   NaN                                                NaN                 NaN                               NaN                     1.0  [severity: warning, message: failed to retriev...```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df if available in the kernel, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Compute per (cmdb_id, kpi_name): count, mean, global P95
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    mean='mean',
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Keep columns compact and round numeric values for readability
grp['mean'] = grp['mean'].round(6)
grp['p95'] = grp['p95'].round(6)

# Select top 20 groups by P95 descending
top20_p95 = grp.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

top20_p95

```
Out[2]:
```
```python
summary = (
    "Summary of top 20 (by global P95) metric series from metric.csv:\n\n"
    "General: All top groups have 30 data points.\n\n"
    "Top entries by P95 (showing cmdb_id / kpi_name — count, mean, P95):\n"
    "1) adservice2 / runtime.java_lang_Runtime_StartTime — 30, mean 1.647822e+12, P95 1.647822e+12\n"
    "2) adservice  / runtime.java_lang_Runtime_StartTime — 30, mean 1.647523e+12, P95 1.647523e+12\n"
    "3) adservice  / runtime.java_lang_OperatingSystem_ProcessCpuTime — 30, mean 4.231960e+11, P95 4.251735e+11\n"
    "4) adservice2 / runtime.java_lang_OperatingSystem_ProcessCpuTime — 30, mean 2.219045e+11, P95 2.243340e+11\n"
    "5) adservice2 / runtime.java_lang_Threading_CurrentThreadCpuTime — 30, mean 1.644349e+10, P95 1.671253e+10\n"
    "6) adservice2 / runtime.java_lang_Threading_CurrentThreadUserTime — 30, mean 1.529917e+10, P95 1.555112e+10\n"
    "7) adservice2 / runtime.java_lang_Threading_CurrentThreadAllocatedBytes — 30, mean 1.159229e+10, P95 1.178311e+10\n"
    "8) node-4     / system.disk.total — 30, mean 7.616300e+09, P95 7.616300e+09\n"
    "9) adservice  / runtime.java_lang_Threading_CurrentThreadCpuTime — 30, mean 6.473109e+09, P95 6.720741e+09\n"
    "10) adservice / runtime.java_lang_Threading_CurrentThreadUserTime — 30, mean 6.229111e+09, P95 6.467208e+09\n\n"
    "Observations: The top-ranked series are dominated by JVM/runtime metrics (start time, CPU/thread times, allocated bytes) and several node-level disk/inode metrics (e.g., system.disk.total, system.fs.inodes.total)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count          mean           p95
0   adservice2                runtime.java_lang_Runtime_StartTime     30  1.647822e+12  1.647822e+12
1    adservice                runtime.java_lang_Runtime_StartTime     30  1.647523e+12  1.647523e+12
2    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  4.231960e+11  4.251735e+11
3   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  2.219045e+11  2.243340e+11
4   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime     30  1.644349e+10  1.671253e+10
5   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime     30  1.529917e+10  1.555112e+10
6   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...     30  1.159229e+10  1.178311e+10
7       node-4                                  system.disk.total     30  7.616300e+09  7.616300e+09
8    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime     30  6.473109e+09  6.720741e+09
9    adservice  runtime.java_lang_Threading_CurrentThreadUserTime     30  6.229111e+09  6.467208e+09
10      node-4                             system.fs.inodes.total     30  6.447526e+09  6.449805e+09
11      node-4                              system.fs.inodes.free     30  6.444931e+09  6.447210e+09
12      node-3                                  system.disk.total     30  5.867011e+09  5.867011e+09
13      node-1                                  system.disk.total     30  5.867011e+09  5.867011e+09
14      node-2                                  system.disk.total     30  5.867011e+09  5.867011e+09
15      node-4                                   system.disk.used     30  5.808046e+09  5.808544e+09
16      node-6                                  system.disk.total     30  5.283914e+09  5.283914e+09
17   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...     30  4.843049e+09  5.033874e+09
18      node-2                             system.fs.inodes.total     30  4.845524e+09  4.847143e+09
19      node-1                             system.fs.inodes.total     30  4.845519e+09  4.847143e+09```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if present
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'grp' not in globals():
    # compute global P95 per (cmdb_id, kpi_name) as required by rule 9
    grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        mean='mean',
        p95=lambda x: x.quantile(0.95)
    ).reset_index()

tz = pytz.timezone('Asia/Shanghai')

# incident window in local timezone (UTC+8)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
# convert to UTC epoch seconds to match metric_df timestamp (which we treat as epoch seconds UTC)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# filter metric_df to window
window_df = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# merge global P95 into the windowed rows
p95_map = grp[['cmdb_id', 'kpi_name', 'p95']]
window_with_p95 = window_df.merge(p95_map, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

# drop rows without a precomputed global P95 (cannot evaluate anomalies)
window_with_p95 = window_with_p95.dropna(subset=['p95'])

# mark anomaly points where value >= global P95
window_with_p95['is_anomaly'] = (window_with_p95['value'] >= window_with_p95['p95']).astype(int)

# aggregate per group
agg = window_with_p95.groupby(['cmdb_id', 'kpi_name']).agg(
    window_count=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    first_anomaly_ts_raw=('timestamp', lambda s: s[window_with_p95.loc[s.index,'is_anomaly']==1].min() if (window_with_p95.loc[s.index,'is_anomaly']==1).any() else np.nan),
    max_value_in_window=('value','max'),
    p95=('p95','first')
).reset_index()

# compute severity safely (avoid division by zero)
agg['severity'] = np.where(agg['p95']!=0, (agg['max_value_in_window'] - agg['p95'])/agg['p95'], np.nan)

# convert first_anomaly_ts to Asia/Shanghai formatted string, keep compact
def fmt_ts(ts):
    if pd.isna(ts):
        return pd.NA
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

agg['first_anomaly_ts'] = agg['first_anomaly_ts_raw'].apply(fmt_ts)

# keep rows with anomaly_count > 0
anoms = agg[agg['anomaly_count'] > 0].copy()

# round numeric columns for compact display
anoms['p95'] = anoms['p95'].round(6)
anoms['max_value_in_window'] = anoms['max_value_in_window'].round(6)
anoms['severity'] = anoms['severity'].round(6)

# sort by anomaly_count desc then severity desc, limit to top 20
result = anoms.sort_values(['anomaly_count','severity'], ascending=[False, False]).head(20)

# select and order columns for output
final_cols = ['cmdb_id','kpi_name','window_count','anomaly_count','first_anomaly_ts','p95','max_value_in_window','severity']
result = result[final_cols].reset_index(drop=True)

result

```
Out[3]:
```
Summary (window = 2022-03-21 21:30:00 → 2022-03-21 21:59:00 UTC+8):

- Returned groups with anomalies: 20 groups (top 20 by anomaly_count then severity). All listed groups have window_count = 30 (one point per minute for 30 minutes).
- Most groups show anomaly_count = 30 (every point in the window >= the global P95). Many of these have severity = 0.0 (max value equals the P95), meaning values in-window are at or exactly at the global 95th percentile.
- Highest severity and top 3 groups:
  1) node-2 / system.udp.connect.num — window_count 30, anomaly_count 30, first_anomaly_ts 2022-03-21 21:30:00 +0800, P95 = 10.0, max = 11.0, severity = 0.10 (largest relative exceedance).
  2) productcatalogservice2-0 / container.node-6.container_file_descriptors — 30, 30, first at 21:30, P95 = 9.0, max ≈ 9.333333, severity ≈ 0.037037.
  3) adservice2 / runtime.jvm_threads_state.RUNNABLE — 30, 30, first at 21:30, P95 = 5.0, max ≈ 5.083333, severity ≈ 0.016667.
- Remainder of the top 20: dominated by adservice JVM/runtime metrics and various container/node metrics. Many entries have anomaly_count 30 and severity 0.0 (e.g., app.grpc.rr, app.http.count, several JVM classloading and GC metrics), indicating they equal or barely exceed their global P95 but do not substantially exceed it.
- First anomalies occur at the start of the window (most show 2022-03-21 21:30:00 +0800).

Interpretation: during the incident window, several series were at or above their global P95 for the entire 30-minute window. The most notable relative exceedance is node-2's UDP connect count (10 → 11, severity 10%), followed by small exceedances in file descriptor and thread-related JVM metrics. Overall anomalies are concentrated in adservice-related metrics and a few node/container-level metrics.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  window_count  anomaly_count           first_anomaly_ts           p95  max_value_in_window  severity
0                     node-2                             system.udp.connect.num            30             30  2022-03-21 21:30:00 +0800  1.000000e+01         1.100000e+01  0.100000
1   productcatalogservice2-0        container.node-6.container_file_descriptors            30             30  2022-03-21 21:30:00 +0800  9.000000e+00         9.333333e+00  0.037037
2                 adservice2                 runtime.jvm_threads_state.RUNNABLE            30             30  2022-03-21 21:30:00 +0800  5.000000e+00         5.083333e+00  0.016667
3    productcatalogservice-2   container.node-5.container_fs_usage_MB./dev/vda1            30             30  2022-03-21 21:30:00 +0800  1.601560e-01         1.601560e-01  0.000002
4                  adservice                                        app.grpc.rr            30             30  2022-03-21 21:30:00 +0800  1.000000e+02         1.000000e+02  0.000000
5                  adservice                                     app.http.count            30             30  2022-03-21 21:30:00 +0800  2.400000e+01         2.400000e+01  0.000000
6                  adservice                                        app.http.rr            30             30  2022-03-21 21:30:00 +0800  1.000000e+02         1.000000e+02  0.000000
7                  adservice                                        app.http.sr            30             30  2022-03-21 21:30:00 +0800  1.000000e+02         1.000000e+02  0.000000
8                  adservice    runtime.java_lang_ClassLoading_LoadedClassCount            30             30  2022-03-21 21:30:00 +0800  5.175000e+03         5.175000e+03  0.000000
9                  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...            30             30  2022-03-21 21:30:00 +0800  5.201000e+03         5.201000e+03  0.000000
10                 adservice  runtime.java_lang_ClassLoading_UnloadedClassCount            30             30  2022-03-21 21:30:00 +0800  2.600000e+01         2.600000e+01  0.000000
11                 adservice  runtime.java_lang_Compilation_CompilationTimeM...            30             30  2022-03-21 21:30:00 +0800  1.000000e+00         1.000000e+00  0.000000
12                 adservice  runtime.java_lang_GarbageCollector_CollectionC...            30             30  2022-03-21 21:30:00 +0800  5.000000e+00         5.000000e+00  0.000000
13                 adservice  runtime.java_lang_GarbageCollector_CollectionT...            30             30  2022-03-21 21:30:00 +0800  1.450000e+02         1.450000e+02  0.000000
14                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  1.000000e+00         1.000000e+00  0.000000
15                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  1.000000e+00         1.000000e+00  0.000000
16                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  4.200000e+01         4.200000e+01  0.000000
17                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  3.323549e+08         3.323549e+08  0.000000
18                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  5.000000e+00         5.000000e+00  0.000000
19                 adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30             30  2022-03-21 21:30:00 +0800  2.365850e+07         2.365850e+07  0.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df if present, otherwise load
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# compute global P95, count, mean per (cmdb_id, trace_name)
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count='count',
    mean='mean',
    p95=lambda x: x.quantile(0.95)
).reset_index()

# round numeric columns for compactness
trace_grp['mean'] = trace_grp['mean'].round(6)
trace_grp['p95'] = trace_grp['p95'].round(6)

# select top 20 by P95 descending
top20_trace_p95 = trace_grp.sort_values('p95', ascending=False).head(20).reset_index(drop=True)

top20_trace_p95

```
Out[4]:
```
```python
summary = (
    "Summary of top 20 trace series by global P95 (computed over full trace.csv):\n\n"
    "General: Each top group has 30 points (full series length in the dataset).\n\n"
    "Top entries by P95 (cmdb_id / trace_name — count, mean, P95):\n"
    "1) frontend2-0 / trace.self.row_count — 30, mean 670.5, P95 836.40\n"
    "2) frontend-2  / trace.self.row_count — 30, mean 557.33, P95 767.80\n"
    "3) frontend-1  / trace.self.row_count — 30, mean 456.43, P95 576.50\n"
    "4) frontend-0  / trace.self.row_count — 30, mean 228.63, P95 459.55\n"
    "5) productcatalogservice2-0 / trace.from_frontend2-0.row_count — 30, mean 310.43, P95 404.00\n"
    "6) frontend2-0 / trace.to_productcatalogservice2-0.row_count — 30, mean 310.43, P95 404.00\n\n"
    "Observations: The highest P95 values are dominated by frontend 'row_count' metrics (self counts), followed by traces representing traffic between frontend and productcatalogservice (from/to patterns). This indicates heavy request/row-count activity at the frontends and corresponding load toward the productcatalogservice in the dataset."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  count        mean     p95
0                frontend2-0                         trace.self.row_count     30  670.500000  836.40
1                 frontend-2                         trace.self.row_count     30  557.333333  767.80
2                 frontend-1                         trace.self.row_count     30  456.433333  576.50
3                 frontend-0                         trace.self.row_count     30  228.633333  459.55
4   productcatalogservice2-0             trace.from_frontend2-0.row_count     30  310.433333  404.00
5                frontend2-0  trace.to_productcatalogservice2-0.row_count     30  310.433333  404.00
6         currencyservice2-0             trace.from_frontend2-0.row_count     30  196.300000  253.10
7                frontend2-0        trace.to_currencyservice2-0.row_count     30  196.300000  253.10
8                 frontend-2   trace.to_productcatalogservice-1.row_count     30   85.833333  118.10
9    productcatalogservice-1              trace.from_frontend-2.row_count     30   85.833333  118.10
10                frontend-2   trace.to_productcatalogservice-0.row_count     30   85.800000  118.10
11   productcatalogservice-0              trace.from_frontend-2.row_count     30   85.800000  118.10
12                frontend-2   trace.to_productcatalogservice-2.row_count     30   85.800000  117.65
13   productcatalogservice-2              trace.from_frontend-2.row_count     30   85.800000  117.65
14   productcatalogservice-1              trace.from_frontend-1.row_count     30   70.500000   92.20
15                frontend-1   trace.to_productcatalogservice-1.row_count     30   70.500000   92.20
16                frontend-1   trace.to_productcatalogservice-0.row_count     30   70.466667   91.65
17   productcatalogservice-0              trace.from_frontend-1.row_count     30   70.466667   91.65
18                frontend-1   trace.to_productcatalogservice-2.row_count     30   70.466667   91.20
19   productcatalogservice-2              trace.from_frontend-1.row_count     30   70.466667   91.20```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df if available
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# ensure timezone object
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 (and count, mean) per (cmdb_id, trace_name) using full series
trace_thresh = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count='count',
    mean='mean',
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()

# 2) Filter trace_df to the incident window 2022-03-21 21:30:00 → 2022-03-21 21:59:00 (UTC+8)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

window_df = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

# 3) Merge global thresholds into windowed data
window_with_thresh = window_df.merge(trace_thresh[['cmdb_id','trace_name','p95','p5']], on=['cmdb_id','trace_name'], how='left', validate='m:1')

# drop rows where thresholds are missing (cannot evaluate)
window_with_thresh = window_with_thresh.dropna(subset=['p95','p5'])

# mark high/low anomalies
window_with_thresh['is_high'] = (window_with_thresh['value'] >= window_with_thresh['p95']).astype(int)
window_with_thresh['is_low']  = (window_with_thresh['value'] <= window_with_thresh['p5']).astype(int)

# aggregate basic stats per group within the window
base_agg = window_with_thresh.groupby(['cmdb_id','trace_name']).agg(
    window_count=('value','size'),
    anomaly_count_high=('is_high','sum'),
    anomaly_count_low=('is_low','sum'),
    max_value_in_window=('value','max'),
    min_value_in_window=('value','min'),
    p95=('p95','first'),
    p5=('p5','first')
).reset_index()

# compute first anomaly timestamps for high and low (raw epoch seconds)
first_high = window_with_thresh[window_with_thresh['is_high']==1].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'first_anomaly_high_ts_raw'})
first_low  = window_with_thresh[window_with_thresh['is_low']==1].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'first_anomaly_low_ts_raw'})

# merge first anomaly timestamps into aggregation
agg = base_agg.merge(first_high, on=['cmdb_id','trace_name'], how='left')
agg = agg.merge(first_low, on=['cmdb_id','trace_name'], how='left')

# helper to format epoch -> Asia/Shanghai string
def fmt_ts(ts):
    if pd.isna(ts):
        return pd.NA
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

agg['first_anomaly_high_ts'] = agg['first_anomaly_high_ts_raw'].apply(fmt_ts)
agg['first_anomaly_low_ts']  = agg['first_anomaly_low_ts_raw'].apply(fmt_ts)

# compute severities safely
agg['severity_high'] = np.where(agg['p95']>0, (agg['max_value_in_window'] - agg['p95'])/agg['p95'], np.nan)
agg['severity_low']  = np.where(agg['p5']>0, (agg['p5'] - agg['min_value_in_window'])/agg['p5'], np.nan)

# keep only groups with some anomaly in window
agg_anom = agg[(agg['anomaly_count_high']>0) | (agg['anomaly_count_low']>0)].copy()

# sorting by total anomaly count desc, then by max severity desc
agg_anom['total_anoms'] = agg_anom['anomaly_count_high'] + agg_anom['anomaly_count_low']
agg_anom['max_severity'] = agg_anom[['severity_high','severity_low']].abs().max(axis=1)  # use absolute magnitude for sorting

# round numeric columns for compactness
agg_anom[['p95','p5','max_value_in_window','min_value_in_window','severity_high','severity_low']] = agg_anom[['p95','p5','max_value_in_window','min_value_in_window','severity_high','severity_low']].round(6)

# select and order output columns
out_cols = [
    'cmdb_id','trace_name','window_count',
    'anomaly_count_high','anomaly_count_low',
    'first_anomaly_high_ts','first_anomaly_low_ts',
    'p95','p5','max_value_in_window','min_value_in_window',
    'severity_high','severity_low'
]

result = agg_anom.sort_values(['total_anoms','max_severity'], ascending=[False, False]).head(20)[out_cols].reset_index(drop=True)

result

```
Out[5]:
```
Summary (window = 2022-03-21 21:30 → 21:59 UTC+8):

- Returned top 20 trace groups with anomalies (all shown groups have window_count = 30).
- For every returned group both anomaly_count_high and anomaly_count_low = 30 (every point in the 30‑min window was >= global P95 and <= global P5).
- Many groups have p95 and p5 equal (or extremely small) so max/min in-window equal the thresholds and severity_high/severity_low are 0.0 (e.g., cartservice2-0 trace.duration_p95 and frontend2-0 → cartservice2-0), or thresholds are zero causing severity to be undefined (NaN) for those rows (several adservice and cartservice error_rate entries where p95 and p5 are 0.000).
- First anomalies for both high and low occur at the start of the window (2022-03-21 21:30:00 +0800) for all listed groups.
- Interpretation: these trace series were constant (or tightly clustered) relative to their historical distribution so that all points fall at or equal to both the 95th and 5th percentile. This pattern suggests the reported “anomalies” are not clear spikes/dips but reflect series with very small dynamic range or zero-valued thresholds, making severity metrics uninformative for many groups.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                            trace_name  window_count  anomaly_count_high  anomaly_count_low      first_anomaly_high_ts       first_anomaly_low_ts    p95     p5  max_value_in_window  min_value_in_window  severity_high  severity_low
0   cartservice2-0   trace.from_frontend2-0.duration_p95            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.001  0.001                0.001                0.001            0.0           0.0
1      frontend2-0  trace.to_cartservice2-0.duration_p95            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.001  0.001                0.001                0.001            0.0           0.0
2      adservice-0      trace.from_frontend-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
3      adservice-0      trace.from_frontend-1.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
4      adservice-0      trace.from_frontend-2.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
5      adservice-1      trace.from_frontend-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
6      adservice-1      trace.from_frontend-1.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
7      adservice-1      trace.from_frontend-2.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
8      adservice-2      trace.from_frontend-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
9      adservice-2      trace.from_frontend-1.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
10     adservice-2      trace.from_frontend-2.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
11    adservice2-0     trace.from_frontend2-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
12   cartservice-0      trace.from_frontend-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
13   cartservice-0      trace.from_frontend-1.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
14   cartservice-0      trace.from_frontend-2.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
15   cartservice-0              trace.self.duration_mean            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
16   cartservice-0               trace.self.duration_p95            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
17   cartservice-0                 trace.self.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
18   cartservice-1      trace.from_frontend-0.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN
19   cartservice-1      trace.from_frontend-1.error_rate            30                  30                 30  2022-03-21 21:30:00 +0800  2022-03-21 21:30:00 +0800  0.000  0.000                0.000                0.000            NaN           NaN```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse error_logs_df if available, else load
if 'error_logs_df' not in globals():
    error_logs_df = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

# define incident window in UTC+8 and convert to epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:59', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# filter to window (inclusive)
errs_window = error_logs_df[(error_logs_df['timestamp'] >= start_epoch) & (error_logs_df['timestamp'] <= end_epoch)].copy()

# Helper to format epoch -> Asia/Shanghai string
def fmt_ts_epoch(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %z')

# 1) Top 20 timestamps by message count: timestamp, count, up to 3 distinct cmdb_id
ts_group = (
    errs_window.groupby('timestamp', as_index=False)
    .agg(count=('message','size'), cmdb_ids=('cmdb_id', lambda s: list(pd.unique(s))))
)
# create compact sample of up to 3 cmdb_ids as string
ts_group['cmdb_sample'] = ts_group['cmdb_ids'].apply(lambda lst: ', '.join(lst[:3]) if isinstance(lst, (list,tuple)) else '')
# format timestamp
ts_group['timestamp_str'] = fmt_ts_epoch(ts_group['timestamp'])
top_timestamps_df = ts_group.sort_values('count', ascending=False).head(20)[['timestamp_str','count','cmdb_sample']].reset_index(drop=True)

# 2) For each cmdb_id present in error_logs within window: total count, earliest ts, latest ts
cmdb_grp = (
    errs_window.groupby('cmdb_id', as_index=False)
    .agg(total_count=('message','size'),
         earliest_ts_raw=('timestamp','min'),
         latest_ts_raw=('timestamp','max'))
)
cmdb_grp['earliest_ts'] = fmt_ts_epoch(cmdb_grp['earliest_ts_raw'])
cmdb_grp['latest_ts']   = fmt_ts_epoch(cmdb_grp['latest_ts_raw'])
cmdb_summary_df = cmdb_grp[['cmdb_id','total_count','earliest_ts','latest_ts']].sort_values('total_count', ascending=False).reset_index(drop=True)

# 3) Up to 10 distinct sample messages across the full file (shortened to 200 chars each)
all_messages = error_logs_df['message'].dropna().astype(str).unique()
sample_messages = all_messages[:10]
# shorten to 200 chars and append ellipsis if truncated
def shorten(msg, n=200):
    if len(msg) > n:
        return msg[:n-3] + '...'
    return msg
message_samples_df = pd.DataFrame({
    'sample_idx': range(1, len(sample_messages)+1),
    'message_sample': [shorten(m,200) for m in sample_messages]
})

# Keep outputs compact: present the three DataFrames
top_timestamps_df, cmdb_summary_df, message_samples_df

```
Out[6]:
```
Summary (error_logs within 2022-03-21 21:30:00 → 2022-03-21 21:59:59 UTC+8):

1) Top 20 timestamps by message count (timestamp — count — up to 3 cmdb_id involved)
- 2022-03-21 21:36:54 +0800 — 3 — frontend-2, frontend-1, frontend-0
- 2022-03-21 21:37:17 +0800 — 3 — frontend-2, frontend-0, frontend-1
- 2022-03-21 21:43:42 +0800 — 3 — frontend-2, frontend-1, frontend-0
- 2022-03-21 21:31:19 +0800 — 3 — frontend-2, frontend-0, frontend-1
- 2022-03-21 21:34:28 +0800 — 3 — frontend-1, frontend-0, frontend-2
- 2022-03-21 21:36:08 +0800 — 3 — frontend-1, frontend-0, frontend-2
- 2022-03-21 21:31:54 +0800 — 3 — frontend-2, frontend-1, frontend-0
- 2022-03-21 21:59:19 +0800 — 3 — frontend-2, frontend-0, frontend-1
- 2022-03-21 21:53:32 +0800 — 3 — frontend-1, frontend-0, frontend-2
- 2022-03-21 21:35:12 +0800 — 2 — frontend-0, frontend-1
- 2022-03-21 21:35:31 +0800 — 2 — frontend-2, frontend-1
- 2022-03-21 21:35:37 +0800 — 2 — frontend-2, frontend-0
- 2022-03-21 21:35:44 +0800 — 2 — frontend-0, frontend-2
- 2022-03-21 21:52:04 +0800 — 2 — frontend-1, frontend-2
- 2022-03-21 21:35:57 +0800 — 2 — frontend-2, frontend-1
- 2022-03-21 21:51:44 +0800 — 2 — frontend-2, frontend-1
- 2022-03-21 21:36:15 +0800 — 2 — frontend-2, frontend-1
- 2022-03-21 21:36:33 +0800 — 2 — frontend-0, frontend-2
- 2022-03-21 21:36:41 +0800 — 2 — frontend-0, frontend-2
- 2022-03-21 21:35:20 +0800 — 2 — frontend-0, frontend-1

2) Per-cmdb_id summary (within window)
- frontend-2 — total messages: 334; earliest: 2022-03-21 21:30:01 +0800; latest: 2022-03-21 21:59:54 +0800
- frontend-1 — total messages: 272; earliest: 2022-03-21 21:30:00 +0800; latest: 2022-03-21 21:59:55 +0800
- frontend-0 — total messages: 137; earliest: 2022-03-21 21:30:00 +0800; latest: 2022-03-21 21:59:51 +0800

3) Sample error messages (up to 10, each truncated to 200 chars)
1) "severity: warning, message: failed to retrieve..."

The original code execution output of IPython Kernel is also provided below for reference:

(                timestamp_str  count                         cmdb_sample
0   2022-03-21 21:36:54 +0800      3  frontend-2, frontend-1, frontend-0
1   2022-03-21 21:37:17 +0800      3  frontend-2, frontend-0, frontend-1
2   2022-03-21 21:43:42 +0800      3  frontend-2, frontend-1, frontend-0
3   2022-03-21 21:31:19 +0800      3  frontend-2, frontend-0, frontend-1
4   2022-03-21 21:34:28 +0800      3  frontend-1, frontend-0, frontend-2
5   2022-03-21 21:36:08 +0800      3  frontend-1, frontend-0, frontend-2
6   2022-03-21 21:31:54 +0800      3  frontend-2, frontend-1, frontend-0
7   2022-03-21 21:59:19 +0800      3  frontend-2, frontend-0, frontend-1
8   2022-03-21 21:53:32 +0800      3  frontend-1, frontend-0, frontend-2
9   2022-03-21 21:35:12 +0800      2              frontend-0, frontend-1
10  2022-03-21 21:35:31 +0800      2              frontend-2, frontend-1
11  2022-03-21 21:35:37 +0800      2              frontend-2, frontend-0
12  2022-03-21 21:35:44 +0800      2              frontend-0, frontend-2
13  2022-03-21 21:52:04 +0800      2              frontend-1, frontend-2
14  2022-03-21 21:35:57 +0800      2              frontend-2, frontend-1
15  2022-03-21 21:51:44 +0800      2              frontend-2, frontend-1
16  2022-03-21 21:36:15 +0800      2              frontend-2, frontend-1
17  2022-03-21 21:36:33 +0800      2              frontend-0, frontend-2
18  2022-03-21 21:36:41 +0800      2              frontend-0, frontend-2
19  2022-03-21 21:35:20 +0800      2              frontend-0, frontend-1,       cmdb_id  total_count                earliest_ts                  latest_ts
0  frontend-2          334  2022-03-21 21:30:01 +0800  2022-03-21 21:59:54 +0800
1  frontend-1          272  2022-03-21 21:30:00 +0800  2022-03-21 21:59:55 +0800
2  frontend-0          137  2022-03-21 21:30:00 +0800  2022-03-21 21:59:51 +0800,    sample_idx                                     message_sample
0           1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse existing log_df if present, else load
if 'log_df' not in globals():
    log_df = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, log_name) using full series (rule 9)
log_thresh = log_df.groupby(['cmdb_id','log_name'])['value'].agg(
    count='count',
    mean='mean',
    p95=lambda x: x.quantile(0.95)
).reset_index()

# 2) Define incident window (UTC+8) and convert to epoch seconds UTC
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# filter log.csv to window
window_logs = log_df[(log_df['timestamp'] >= start_epoch) & (log_df['timestamp'] <= end_epoch)].copy()

# 3) Merge global P95 into windowed rows
window_with_p95 = window_logs.merge(log_thresh[['cmdb_id','log_name','p95']], on=['cmdb_id','log_name'], how='left', validate='m:1')
window_with_p95 = window_with_p95.dropna(subset=['p95'])  # ensure groups have global P95

# mark anomalies where value >= global P95
window_with_p95['is_anomaly'] = (window_with_p95['value'] >= window_with_p95['p95']).astype(int)

# aggregate per group within window
base_agg = window_with_p95.groupby(['cmdb_id','log_name']).agg(
    window_count=('value','size'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max'),
    p95=('p95','first')
).reset_index()

# compute first anomaly timestamp (raw epoch seconds) for groups with anomalies
first_anom = window_with_p95[window_with_p95['is_anomaly']==1].groupby(['cmdb_id','log_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'first_anomaly_ts_raw'})

# merge first anomaly ts
agg = base_agg.merge(first_anom, on=['cmdb_id','log_name'], how='left')

# compute severity safely (P95>0)
agg['severity'] = np.where(agg['p95']>0, (agg['max_value_in_window'] - agg['p95'])/agg['p95'], np.nan)

# format first_anomaly_ts to Asia/Shanghai string
def fmt_ts(val):
    if pd.isna(val):
        return pd.NA
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

agg['first_anomaly_ts'] = agg['first_anomaly_ts_raw'].apply(fmt_ts)

# keep only groups with anomaly_count > 0
agg_anom = agg[agg['anomaly_count'] > 0].copy()

# sort by first_anomaly_ts_raw ascending (earliest) then severity descending, limit to top 20
agg_anom_sorted = agg_anom.sort_values(by=['first_anomaly_ts_raw','severity'], ascending=[True, False]).head(20)

# round numeric columns for compact display
agg_anom_sorted[['p95','max_value_in_window','severity']] = agg_anom_sorted[['p95','max_value_in_window','severity']].round(6)

# select requested output columns
result = agg_anom_sorted[[
    'cmdb_id','log_name','window_count','anomaly_count','first_anomaly_ts','p95','max_value_in_window','severity'
]].reset_index(drop=True)

result

```
Out[7]:
```
```python
# Compose a plain-English summary of the previous log.csv anomaly results (variable `result` expected in kernel)
import pandas as pd
import numpy as np
import pytz

# reuse 'result' from previous cell
if 'result' not in globals():
    raise RuntimeError("Expected DataFrame 'result' from prior computation is missing.")

tz = pytz.timezone('Asia/Shanghai')

# Basic stats
total_groups = int(result.shape[0])
nan_severity_count = int(result['severity'].isna().sum())
# parse earliest first_anomaly_ts (ignore missing)
valid_ts = result['first_anomaly_ts'].dropna()
earliest_ts = valid_ts.min() if not valid_ts.empty else "N/A"

# Prepare top 5 entries for brief listing
top5 = result.head(5).copy()
def fmt_row(r):
    sev = r['severity']
    sev_str = f"{sev:.6f}" if pd.notna(sev) else "NaN"
    return f"{r['cmdb_id']} / {r['log_name']} — window_count={int(r['window_count'])}, anomaly_count={int(r['anomaly_count'])}, first_anomaly_ts={r['first_anomaly_ts']}, p95={r['p95']}, max={r['max_value_in_window']}, severity={sev_str}"

top5_lines = [fmt_row(row) for _, row in top5.iterrows()]

summary = (
    "Plain-English summary of log.csv anomalies in window 2022-03-21 21:30:00 → 2022-03-21 21:59:00 (UTC+8):\n\n"
    f"- Groups with anomalies returned (limited view): {total_groups}\n"
    f"- Earliest reported anomaly timestamp among these groups: {earliest_ts}\n"
    f"- {nan_severity_count} group(s) have undefined (NaN) severity because P95 was zero.\n\n"
    "Top 5 anomaly groups (cmdb_id / log_name — window_count, anomaly_count, first_anomaly_ts, p95, max, severity):\n"
    + "\n".join([f"{i+1}) {line}" for i, line in enumerate(top5_lines)])
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  window_count  anomaly_count           first_anomaly_ts     p95  max_value_in_window  severity
0               adservice-2    log.row_count            30              4  2022-03-21 21:30:00 +0800   84.00                 88.0  0.047619
1   recommendationservice-0    log.row_count            30              2  2022-03-21 21:30:00 +0800  143.10                149.0  0.041230
2             cartservice-2    log.row_count            30              2  2022-03-21 21:30:00 +0800  554.50                572.0  0.031560
3   recommendationservice-1    log.row_count            30              2  2022-03-21 21:30:00 +0800  144.55                149.0  0.030785
4             cartservice-0    log.row_count            30              2  2022-03-21 21:30:00 +0800  544.60                561.0  0.030114
5   productcatalogservice-1    log.row_count            30              2  2022-03-21 21:30:00 +0800  669.10                686.0  0.025258
6   productcatalogservice-0    log.row_count            30              2  2022-03-21 21:30:00 +0800  669.70                681.0  0.016873
7   recommendationservice-2    log.row_count            30              2  2022-03-21 21:30:00 +0800  145.10                147.0  0.013094
8             cartservice-1    log.row_count            30              2  2022-03-21 21:30:00 +0800  544.95                550.0  0.009267
9                frontend-1  log.error_count            30              6  2022-03-21 21:30:00 +0800   12.00                 12.0  0.000000
10             redis-cart-0    log.row_count             6              5  2022-03-21 21:30:00 +0800    5.00                  5.0  0.000000
11              adservice-0  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
12              adservice-1  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
13              adservice-2  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
14            cartservice-0  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
15            cartservice-1  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
16            cartservice-2  log.error_count            30             30  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
17        checkoutservice-0  log.error_count            28             28  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
18        checkoutservice-1  log.error_count            25             25  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN
19        checkoutservice-2  log.error_count            28             28  2022-03-21 21:30:00 +0800    0.00                  0.0       NaN```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse error_logs_df if present
if 'error_logs_df' not in globals():
    error_logs_df = pd.read_csv("error_logs.csv")

tz = pytz.timezone('Asia/Shanghai')

# define incident window and epoch bounds (UTC)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:59', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# filter to window (inclusive)
errs_window = error_logs_df[(error_logs_df['timestamp'] >= start_epoch) & (error_logs_df['timestamp'] <= end_epoch)].copy()

# floor timestamps to minute (epoch seconds)
errs_window['minute_epoch'] = (errs_window['timestamp'] // 60) * 60

# helper to format minute epoch to Asia/Shanghai minute string
def fmt_minute_epoch(e):
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:00 %z')

# helper to shorten messages to 200 chars
def shorten(msg, n=200):
    msg = str(msg)
    return msg if len(msg) <= n else msg[:n-3] + '...'

# aggregate by minute
agg = errs_window.groupby('minute_epoch').agg(
    total_message_count=('message','size'),
    count_frontend_0=('cmdb_id', lambda s: int((s=='frontend-0').sum())),
    count_frontend_1=('cmdb_id', lambda s: int((s=='frontend-1').sum())),
    count_frontend_2=('cmdb_id', lambda s: int((s=='frontend-2').sum())),
    message_samples=('message', lambda s: list(pd.unique(s.dropna())[:3]))
).reset_index()

# format minute timestamp and message samples as joined shortened strings
agg['minute_timestamp'] = agg['minute_epoch'].apply(fmt_minute_epoch)
agg['message_sample_snippets'] = agg['message_samples'].apply(lambda lst: ' | '.join(shorten(m) for m in lst))

# rename frontend count columns to requested names with hyphens
agg = agg.rename(columns={
    'count_frontend_0': 'count_frontend-0',
    'count_frontend_1': 'count_frontend-1',
    'count_frontend_2': 'count_frontend-2'
})

# select and order columns
minute_summary = agg[['minute_timestamp','total_message_count','count_frontend-0','count_frontend-1','count_frontend-2','message_sample_snippets']]

# sort by total_message_count desc then minute_timestamp asc, limit top 10
top10_minutes = minute_summary.sort_values(['total_message_count','minute_timestamp'], ascending=[False, True]).head(10).reset_index(drop=True)

top10_minutes

```
Out[8]:
```
```python
summary = (
    "Error-logs aggregated by minute (window 2022-03-21 21:30:00 → 21:59:59 UTC+8):\n\n"
    "Top 10 minutes (minute — total_messages — frontend-0, frontend-1, frontend-2):\n"
    "1) 2022-03-21 21:32:00 +0800 — 35 — (7, 12, 16)\n"
    "2) 2022-03-21 21:34:00 +0800 — 35 — (8, 12, 15)\n"
    "3) 2022-03-21 21:30:00 +0800 — 34 — (8, 12, 14)\n"
    "4) 2022-03-21 21:31:00 +0800 — 33 — (10, 7, 16)\n"
    "5) 2022-03-21 21:36:00 +0800 — 33 — (10, 10, 13)\n"
    "6) 2022-03-21 21:35:00 +0800 — 32 — (9, 12, 11)\n"
    "7) 2022-03-21 21:38:00 +0800 — 32 — (7, 10, 15)\n"
    "8) 2022-03-21 21:45:00 +0800 — 32 — (4, 11, 17)\n"
    "9) 2022-03-21 21:40:00 +0800 — 30 — (5, 10, 15)\n"
    "10) 2022-03-21 21:33:00 +0800 — 29 — (8, 8, 13)\n\n"
    "Sample message snippets seen in these minutes (up to 3 per minute):\n"
    "- \"severity: warning, message: failed to retrieve...\" (recurring across many minutes)\n\n"
    "Interpretation: The highest error-log volumes occur in the first half of the window (peaks ~21:30–21:36 and a later spike at 21:45). All three frontend instances (frontend-0, -1, -2) contribute to these peaks. The messages are largely the same recurring warning about failed retrievals, indicating a repeated issue affecting frontends rather than isolated one-off messages."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

minute_timestamp  total_message_count  count_frontend-0  count_frontend-1  count_frontend-2                            message_sample_snippets
0  2022-03-21 21:32:00 +0800                   35                 7                12                16  severity: warning, message: failed to retrieve...
1  2022-03-21 21:34:00 +0800                   35                 8                12                15  severity: warning, message: failed to retrieve...
2  2022-03-21 21:30:00 +0800                   34                 8                12                14  severity: warning, message: failed to retrieve...
3  2022-03-21 21:31:00 +0800                   33                10                 7                16  severity: warning, message: failed to retrieve...
4  2022-03-21 21:36:00 +0800                   33                10                10                13  severity: warning, message: failed to retrieve...
5  2022-03-21 21:35:00 +0800                   32                 9                12                11  severity: warning, message: failed to retrieve...
6  2022-03-21 21:38:00 +0800                   32                 7                10                15  severity: warning, message: failed to retrieve...
7  2022-03-21 21:45:00 +0800                   32                 4                11                17  severity: warning, message: failed to retrieve...
8  2022-03-21 21:40:00 +0800                   30                 5                10                15  severity: warning, message: failed to retrieve...
9  2022-03-21 21:33:00 +0800                   29                 8                 8                13  severity: warning, message: failed to retrieve...```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# reuse existing variables when available
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'grp' not in globals():
    # compute global P95 per (cmdb_id, kpi_name) if missing
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(p95=lambda x: x.quantile(0.95)).reset_index()

tz = pytz.timezone('Asia/Shanghai')

# 1) Filter metric.csv to the incident window (UTC+8) and convert to epoch seconds (UTC)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

window_df = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# 2) Merge global P95 (computed from full series) into the windowed data
p95_map = grp[['cmdb_id','kpi_name','p95']]
window_with_p95 = window_df.merge(p95_map, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Drop groups without a global P95 (can't evaluate)
window_with_p95 = window_with_p95.dropna(subset=['p95'])

# 2) Find earliest timestamp in window where value >= global P95 for each series
anomaly_points = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()

# If no anomalies, produce empty structured result
if anomaly_points.empty:
    top10_minutes = pd.DataFrame(columns=['minute_timestamp','count_of_series_starting_that_minute','sample_cmdb_ids'])
else:
    first_anoms = (
        anomaly_points.groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(first_anomaly_ts_raw=('timestamp','min'))
    )

    # 3) Floor to minute and aggregate by minute
    first_anoms['minute_epoch'] = (first_anoms['first_anomaly_ts_raw'] // 60) * 60

    minute_agg = (
        first_anoms.groupby('minute_epoch', as_index=False)
        .agg(
            count_of_series_starting_that_minute=('cmdb_id','size'),
            sample_cmdb_ids_list=('cmdb_id', lambda s: list(pd.unique(s)[:5]))
        )
    )

    # format minute timestamp to Asia/Shanghai minute string
    minute_agg['minute_timestamp'] = minute_agg['minute_epoch'].apply(
        lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:00 %z')
    )

    # join up to 5 sample cmdb_ids into a compact string
    minute_agg['sample_cmdb_ids'] = minute_agg['sample_cmdb_ids_list'].apply(lambda lst: ', '.join(lst))

    # 4) Return top 10 minutes sorted by count desc then minute_timestamp asc
    top10_minutes = minute_agg.sort_values(['count_of_series_starting_that_minute','minute_timestamp'], ascending=[False, True]) \
                             .head(10)[['minute_timestamp','count_of_series_starting_that_minute','sample_cmdb_ids']].reset_index(drop=True)

# Display compact result
top10_minutes

```
Out[9]:
```
```python
summary = (
    "Summary of series first-crossing the global P95 within the incident window (2022-03-21 21:30:00 → 21:59:00 UTC+8):\n\n"
    "Key findings:\n"
    "- A very large surge of metric series first crossed their global P95 at 2022-03-21 21:30:00 +0800: 3,655 series (sample components: adservice, adservice-0, adservice-1, ...).\n"
    "- After the initial spike, counts of new series crossing P95 drop sharply: 390 series at 21:31, 150 at 21:32, 144 at 21:33.\n"
    "- Additional smaller waves occurred later in the window (examples: 121 series at 21:58, 78 at 21:35, 76 at 21:34, 55 at 21:36 and 21:43, 46 at 21:38).\n"
    "- Sample components repeatedly involved across top minutes include adservice and its pods, cartservice and its pods, and various checkout/productcatalogservice components—indicating the anomaly affected many services/pods concurrently, with the strongest onset at 21:30.\n\n"
    "Interpretation: the incident appears to start sharply at 21:30 (massive simultaneous crossings of P95 across thousands of metric series), followed by smaller, diminishing groups of additional series crossing P95 over the next ~30 minutes. This pattern suggests a broad-impact event beginning at 21:30 that affected many components at once."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

minute_timestamp  count_of_series_starting_that_minute                                    sample_cmdb_ids
0  2022-03-21 21:30:00 +0800                                  3655  adservice, adservice-0, adservice-1, adservice...
1  2022-03-21 21:31:00 +0800                                   390  adservice, adservice-0, cartservice-0, cartser...
2  2022-03-21 21:32:00 +0800                                   150  adservice, adservice-0, adservice-1, adservice...
3  2022-03-21 21:33:00 +0800                                   144  adservice, cartservice-0, cartservice-1, carts...
4  2022-03-21 21:58:00 +0800                                   121  adservice, adservice-0, adservice-1, adservice...
5  2022-03-21 21:35:00 +0800                                    78  adservice, adservice-1, cartservice-1, checkou...
6  2022-03-21 21:34:00 +0800                                    76  adservice-0, adservice2, adservice2-0, cartser...
7  2022-03-21 21:36:00 +0800                                    55  adservice, adservice-1, adservice-2, adservice...
8  2022-03-21 21:43:00 +0800                                    55  adservice, checkoutservice-1, checkoutservice-...
9  2022-03-21 21:38:00 +0800                                    46  adservice, adservice-1, adservice2-0, cartserv...```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'grp' not in globals():
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(p95=lambda x: x.quantile(0.95)).reset_index()

tz = pytz.timezone('Asia/Shanghai')

# Define incident window (UTC+8) and convert to UTC epoch
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# Windowed data
window_df = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# Merge global p95 into windowed rows
p95_map = grp[['cmdb_id','kpi_name','p95']]
window_with_p95 = window_df.merge(p95_map, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# Keep only rows where p95 available
window_with_p95 = window_with_p95.dropna(subset=['p95'])

# Identify anomaly points (value >= p95) within window
anomaly_points = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()

# Earliest anomaly timestamp per series within window
first_anoms = anomaly_points.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    first_anomaly_ts_raw=('timestamp','min')
)

# Target minute epoch for 2022-03-21 21:30:00 +0800 (converted to UTC epoch, floored to minute)
target_min_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
target_epoch_min = int(target_min_local.tz_convert('UTC').timestamp())
target_minute_epoch = (target_epoch_min // 60) * 60

# Add minute_epoch column to first_anoms
first_anoms['minute_epoch'] = (first_anoms['first_anomaly_ts_raw'] // 60) * 60

# Filter series whose first anomaly minute == target minute
first30 = first_anoms[first_anoms['minute_epoch'] == target_minute_epoch].copy()

# Part 1: aggregate by cmdb_id -> top 10 cmdb_id by number of series starting at that minute
if not first30.empty:
    cmdb_agg = (
        first30.groupby('cmdb_id', as_index=False)
        .agg(
            series_count=('kpi_name','size'),
            sample_kpis=('kpi_name', lambda s: list(pd.unique(s)[:5]))
        )
        .sort_values('series_count', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    cmdb_agg['sample_kpi_examples'] = cmdb_agg['sample_kpis'].apply(lambda lst: ', '.join(lst))
    cmdb_result = cmdb_agg[['cmdb_id','series_count','sample_kpi_examples']]
else:
    cmdb_result = pd.DataFrame(columns=['cmdb_id','series_count','sample_kpi_examples'])

# Part 2: compute severity per series = (max_value_in_window - p95)/p95 and return top 10 series by severity
# Compute per-series max_value_in_window and p95 (from merged window_with_p95)
window_stats = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    max_value_in_window=('value','max'),
    p95=('p95','first')
)

# Merge with first30 to get only series that started at target minute
if not first30.empty:
    series_target = first30.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute severity safely (NaN if p95 == 0)
    series_target['severity'] = np.where(series_target['p95']>0,
                                         (series_target['max_value_in_window'] - series_target['p95']) / series_target['p95'],
                                         np.nan)

    # Format first_anomaly_ts to Asia/Shanghai
    def fmt_ts_epoch(val):
        if pd.isna(val):
            return pd.NA
        return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

    series_target['first_anomaly_ts'] = series_target['first_anomaly_ts_raw'].apply(fmt_ts_epoch)

    # Round numeric columns for compactness
    series_target['p95'] = series_target['p95'].round(6)
    series_target['max_value_in_window'] = series_target['max_value_in_window'].round(6)
    series_target['severity'] = series_target['severity'].round(6)

    series_result = series_target.sort_values('severity', ascending=False).head(10)[
        ['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity']
    ].reset_index(drop=True)
else:
    series_result = pd.DataFrame(columns=['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity'])

# Return compact results
cmdb_result, series_result

```
Out[10]:
```
```python
summary = (
    "Summary (series that first crossed global P95 at 2022-03-21 21:30:00 +0800):\n\n"
    "1) Top 10 affected cmdb_id by number of series starting at 21:30 (series_count — sample kpi_names):\n"
    " - adservice — 296 series (examples: app.grpc.count, app.grpc.rr, app.http.count, ...)\n"
    " - adservice2 — 285 series (examples: runtime.java_lang_ClassLoading_LoadedClassCount, ...)\n"
    " - frontend2-0 — 138 series (examples: container.node-6.container_cpu_load_average_10, ...)\n"
    " - frontend-2 — 129 series (examples: container.node-5.container_cpu_load_average_10, ...)\n"
    " - frontend-1 — 108 series (examples: container.node-5.container_cpu_cfs_throttled_percent, ...)\n"
    " - frontend-0 — 102 series (examples: container.node-5.container_cpu_cfs_periods, ...)\n"
    " - adservice2-0 — 95 series (examples: container.node-6.container_cpu_cfs_throttled_percent, ...)\n"
    " - productcatalogservice-2 — 94 series (examples: container.node-5.container_cpu_cfs_periods, ...)\n"
    " - productcatalogservice-0 — 91 series (examples: container.node-5.container_cpu_cfs_periods, ...)\n"
    " - checkoutservice-2 — 87 series (examples: container.node-5.container_cpu_cfs_throttled_percent, ...)\n\n"
    "2) Top 10 individual series (cmdb_id / kpi_name) among those that first crossed at 21:30, "
    "sorted by severity ((max - P95)/P95) descending:\n"
    " 1) adservice-1 / container.node-5.container_network_receive_MB... — first_anomaly_ts 2022-03-21 21:30:00 +0800, p95=0.051882, max=0.588792, severity=10.348682\n"
    " 2) checkoutservice-2 / container.node-5.container_network_receive_MB... — 2022-03-21 21:30:00 +0800, p95=0.050296, max=0.560767, severity=10.149330\n"
    " 3) cartservice-2 / container.node-5.container_network_receive_MB... — 2022-03-21 21:30:00 +0800, p95=0.057798, max=0.576617, severity=8.976414\n"
    " 4) cartservice-1 / container.node-5.container_network_receive_MB... — 2022-03-21 21:30:00 +0800, p95=0.059961, max=0.570898, severity=8.521156\n"
    " 5) adservice2-0 / container.node-6.container_network_receive_MB... — 2022-03-21 21:30:00 +0800, p95=0.076668, max=0.628578, severity=7.198697\n"
    " 6) currencyservice-0 / container.node-5.container_network_receive_MB... — 2022-03-21 21:30:00 +0800, p95=0.099203, max=0.611960, severity=5.168769\n"
    " 7) node-6 / system.io.await — 2022-03-21 21:30:00 +0800, p95=0.635000, max=2.590000, severity=3.078740\n"
    " 8) node-6 / system.io.w_await — 2022-03-21 21:30:00 +0800, p95=0.635000, max=2.590000, severity=3.078740\n"
    " 9) node-4 / system.io.w_s — 2022-03-21 21:30:00 +0800, p95=0.275000, max=1.000000, severity=2.636364\n"
    "10) currencyservice / app.grpc.mrt — 2022-03-21 21:30:00 +0800, p95=0.000292, max=0.000447, severity=0.530252\n\n"
    "Interpretation: A massive number of metric series across many services (notably adservice and frontends) first exceeded their historical P95 at 21:30. "
    "The highest per-series severities are dominated by large relative spikes in container network receive (MB) metrics across multiple services and elevated node I/O await/write-await metrics, indicating a rapid surge in network receive and I/O latency at the same time the many series crossed P95."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                   cmdb_id  series_count                                sample_kpi_examples
0                adservice           296  app.grpc.count, app.grpc.rr, app.http.count, a...
1               adservice2           285  runtime.java_lang_ClassLoading_LoadedClassCoun...
2              frontend2-0           138  container.node-6.container_cpu_load_average_10...
3               frontend-2           129  container.node-5.container_cpu_load_average_10...
4               frontend-1           108  container.node-5.container_cpu_cfs_throttled_p...
5               frontend-0           102  container.node-5.container_cpu_cfs_periods, co...
6             adservice2-0            95  container.node-6.container_cpu_cfs_throttled_p...
7  productcatalogservice-2            94  container.node-5.container_cpu_cfs_periods, co...
8  productcatalogservice-0            91  container.node-5.container_cpu_cfs_periods, co...
9        checkoutservice-2            87  container.node-5.container_cpu_cfs_throttled_p...,              cmdb_id                                           kpi_name           first_anomaly_ts       p95  max_value_in_window   severity
0        adservice-1  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.051882             0.588792  10.348682
1  checkoutservice-2  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.050296             0.560767  10.149330
2      cartservice-2  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.057798             0.576617   8.976414
3      cartservice-1  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.059961             0.570898   8.521156
4       adservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.076668             0.628578   7.198697
5  currencyservice-0  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800  0.099203             0.611960   5.168769
6             node-6                                    system.io.await  2022-03-21 21:30:00 +0800  0.635000             2.590000   3.078740
7             node-6                                  system.io.w_await  2022-03-21 21:30:00 +0800  0.635000             2.590000   3.078740
8             node-4                                      system.io.w_s  2022-03-21 21:30:00 +0800  0.275000             1.000000   2.636364
9    currencyservice                                       app.grpc.mrt  2022-03-21 21:30:00 +0800  0.000292             0.000447   0.530252)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables if available
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'grp' not in globals():
    # global P95 per series (rule 9)
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(p95=lambda x: x.quantile(0.95)).reset_index()

tz = pytz.timezone('Asia/Shanghai')

# incident window bounds (UTC+8 -> convert to UTC epoch)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# reuse windowed data if present, otherwise compute
if 'window_with_p95' not in globals():
    # filter full metrics to window and merge global p95
    window_df = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
    p95_map = grp[['cmdb_id','kpi_name','p95']]
    window_with_p95 = window_df.merge(p95_map, on=['cmdb_id','kpi_name'], how='left', validate='m:1')

# ensure rows with p95 available
window_with_p95 = window_with_p95.dropna(subset=['p95'])

# identify anomaly points (value >= global p95) and earliest anomaly per series
if 'first_anoms' not in globals():
    anomaly_points = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()
    first_anoms = anomaly_points.groupby(['cmdb_id','kpi_name'], as_index=False).agg(first_anomaly_ts_raw=('timestamp','min'))

# target minute epoch (UTC) for 2022-03-21 21:30:00 +0800
target_min_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
target_epoch_min = int(target_min_local.tz_convert('UTC').timestamp())
target_minute_epoch = (target_epoch_min // 60) * 60

# filter series whose first_anomaly minute equals target minute
first_anoms['minute_epoch'] = (first_anoms['first_anomaly_ts_raw'] // 60) * 60
first30 = first_anoms[first_anoms['minute_epoch'] == target_minute_epoch].copy()

# If no series, return empty DataFrame with required columns
if first30.empty:
    result = pd.DataFrame(columns=['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity'])
else:
    # select series whose kpi_name contains network OR receive OR transmit (case-insensitive)
    mask_network = first30['kpi_name'].str.contains('network|receive|transmit', case=False, na=False)
    selected = first30[mask_network].copy()

    if selected.empty:
        result = pd.DataFrame(columns=['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity'])
    else:
        # get global p95 for these series (from grp) and max_value_in_window from the windowed data
        # reuse window_with_p95 and grp
        stats = selected.merge(grp[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
        max_vals = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False).agg(max_value_in_window=('value','max'))
        stats = stats.merge(max_vals, on=['cmdb_id','kpi_name'], how='left')

        # compute severity (NaN if p95 == 0)
        stats['severity'] = np.where(stats['p95']>0, (stats['max_value_in_window'] - stats['p95'])/stats['p95'], np.nan)

        # format first_anomaly_ts to UTC+8 string
        stats['first_anomaly_ts'] = stats['first_anomaly_ts_raw'].apply(
            lambda v: pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')
        )

        # round numeric columns for compactness
        stats['p95'] = stats['p95'].round(6)
        stats['max_value_in_window'] = stats['max_value_in_window'].round(6)
        stats['severity'] = stats['severity'].round(6)

        # select required columns and sort by severity desc, limit top 20
        result = stats[['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity']] \
                    .sort_values('severity', ascending=False).head(20).reset_index(drop=True)

# Display compact result
result

```
Out[11]:
```
Summary (network-related series that first crossed their global P95 at 2022-03-21 21:30:00 +0800):

- Scope: series whose kpi_name contains "network" or "receive" or "transmit" and whose earliest in-window value >= global P95 occurred at 21:30:00 +0800.

- Top observations (sorted by severity, showing cmdb_id / kpi_name — p95 → max — severity):
  1) adservice-1 / container.node-5.container_network_receive_MB... — p95=0.051882 → max=0.588792 — severity=10.3487
  2) checkoutservice-2 / container.node-5.container_network_receive_MB... — p95=0.050296 → max=0.560767 — severity=10.1493
  3) cartservice-2 / container.node-5.container_network_receive_MB... — p95=0.057798 → max=0.576617 — severity=8.9764
  4) cartservice-1 / container.node-5.container_network_receive_MB... — p95=0.059961 → max=0.570898 — severity=8.5212
  5) adservice2-0 / container.node-6.container_network_receive_MB... — p95=0.076668 → max=0.628578 — severity=7.1987
  6) currencyservice-0 / container.node-5.container_network_receive_MB... — p95=0.099203 → max=0.611960 — severity=5.1688

- Additional notes from the list:
  - Several high-severity entries are container_network_receive (MB) for multiple services/pods — very large relative increases vs their historical P95.
  - There are also high relative increases in packet-level metrics and transmit metrics (e.g., container_network_transmit and receive/transmit packets for frontend and other services) with moderate severities (0.1–0.16 for some high-volume packet metrics, and smaller but notable increases for others).
  - Some metrics have very small P95 baselines (leading to large severity ratios even for modest absolute increases).

- Interpretation: at 21:30 the system experienced a simultaneous, large relative surge in container network receive (and related transmit/packet) metrics across many services and pods — strongest on several services’ containers on node-5 and node-6 — consistent with a traffic/network influx or abnormal network I/O affecting many containers at once.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           first_anomaly_ts          p95  max_value_in_window   severity
0               adservice-1  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.051882             0.588792  10.348682
1         checkoutservice-2  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.050296             0.560767  10.149330
2             cartservice-2  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.057798             0.576617   8.976414
3             cartservice-1  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.059961             0.570898   8.521156
4              adservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.076668             0.628578   7.198697
5         currencyservice-0  container.node-5.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.099203             0.611960   5.168769
6         paymentservice2-0  container.node-6.container_network_receive_MB....  2022-03-21 21:30:00 +0800     0.249637             0.377160   0.510833
7               adservice-1  container.node-5.container_network_transmit_pa...  2022-03-21 21:30:00 +0800   349.075000           404.500000   0.158777
8                frontend-2  container.node-5.container_network_transmit_pa...  2022-03-21 21:30:00 +0800  2176.375000          2467.500000   0.133766
9               adservice-1  container.node-5.container_network_transmit_MB...  2022-03-21 21:30:00 +0800     1.083632             1.217581   0.123611
10  recommendationservice-1  container.node-5.container_network_transmit_MB...  2022-03-21 21:30:00 +0800     0.716592             0.804475   0.122640
11               frontend-2  container.node-5.container_network_receive_pac...  2022-03-21 21:30:00 +0800  1619.850000          1816.500000   0.121400
12              adservice-1  container.node-5.container_network_receive_pac...  2022-03-21 21:30:00 +0800   469.700000           525.000000   0.117735
13        paymentservice2-0  container.node-6.container_network_receive_pac...  2022-03-21 21:30:00 +0800   325.333333           362.833333   0.115266
14        checkoutservice-2  container.node-5.container_network_receive_pac...  2022-03-21 21:30:00 +0800   457.050000           506.000000   0.107100
15  recommendationservice-1  container.node-5.container_network_transmit_pa...  2022-03-21 21:30:00 +0800   378.600000           416.500000   0.100106
16  recommendationservice-0  container.node-5.container_network_transmit_pa...  2022-03-21 21:30:00 +0800   361.875000           398.000000   0.099827
17        paymentservice2-0  container.node-6.container_network_transmit_pa...  2022-03-21 21:30:00 +0800   242.141667           264.166667   0.090959
18            cartservice-2  container.node-5.container_network_receive_pac...  2022-03-21 21:30:00 +0800   472.575000           507.000000   0.072846
19            cartservice-1  container.node-5.container_network_transmit_pa...  2022-03-21 21:30:00 +0800   394.225000           418.500000   0.061577```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df and trace_thresh if available
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if 'trace_thresh' not in globals():
    # compute global thresholds (p95) per (cmdb_id, trace_name) if missing
    trace_thresh = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        count='count',
        mean='mean',
        p95=lambda x: x.quantile(0.95)
    ).reset_index()

# Candidate cmdb_ids
candidates = [
    'adservice-1', 'checkoutservice-2', 'cartservice-2', 'cartservice-1',
    'adservice2-0', 'currencyservice-0', 'frontend-2', 'frontend-1', 'frontend-0'
]

tz = pytz.timezone('Asia/Shanghai')

# Incident window (UTC+8) -> convert to UTC epoch seconds
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# Filter trace_df to candidate cmdb_ids and to the incident window
df_cand = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()
df_cand = df_cand[(df_cand['timestamp'] >= start_epoch) & (df_cand['timestamp'] <= end_epoch)].copy()

# derive service name from cmdb_id (part before first '-' e.g., frontend-2 -> frontend)
service_map = {cid: cid.split('-')[0] for cid in candidates}
df_cand['service_name'] = df_cand['cmdb_id'].map(service_map)

# build mask for relevant trace_name patterns:
# - contains 'from_frontend' (incoming from frontend)
# - contains 'trace.self.' (self metrics)
# - contains 'to_{service_name}' (incoming to this service)
lower_trace = df_cand['trace_name'].str.lower()
mask_from_frontend = lower_trace.str.contains('from_frontend', na=False)
mask_self = lower_trace.str.contains('trace.self.', na=False)
# mask for to_<service> per row
mask_to_service = df_cand.apply(lambda r: f"to_{r['service_name']}" in r['trace_name'].lower(), axis=1)

df_filtered = df_cand[mask_from_frontend | mask_self | mask_to_service].copy()

# Merge global p95 thresholds (computed over full trace.csv) - per rule 9 thresholds computed globally first
p95_map = trace_thresh[['cmdb_id','trace_name','p95']]
df_filtered = df_filtered.merge(p95_map, on=['cmdb_id','trace_name'], how='left', validate='m:1')

# Drop rows without global p95
df_filtered = df_filtered.dropna(subset=['p95']).copy()

# Mark high anomalies (value >= global p95)
df_filtered['is_high'] = (df_filtered['value'] >= df_filtered['p95']).astype(int)

# Aggregate per (cmdb_id, trace_name) within window
agg_base = df_filtered.groupby(['cmdb_id','trace_name']).agg(
    window_count=('value','size'),
    anomaly_count_high=('is_high','sum'),
    max_value_in_window=('value','max'),
    p95=('p95','first')
).reset_index()

# Get first anomaly timestamp per group (min timestamp among rows with is_high==1)
first_high = df_filtered[df_filtered['is_high']==1].groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'first_anomaly_ts_raw'})

agg = agg_base.merge(first_high, on=['cmdb_id','trace_name'], how='left')

# Keep only groups with anomaly_count_high > 0
agg = agg[agg['anomaly_count_high'] > 0].copy()

# Compute severity_high safely
agg['severity_high'] = np.where(agg['p95']>0, (agg['max_value_in_window'] - agg['p95'])/agg['p95'], np.nan)

# Format first_anomaly_high_ts to Asia/Shanghai timezone string
def fmt_ts(val):
    if pd.isna(val):
        return pd.NA
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

agg['first_anomaly_high_ts'] = agg['first_anomaly_ts_raw'].apply(fmt_ts)

# Round numeric columns for compact display
agg[['p95','max_value_in_window','severity_high']] = agg[['p95','max_value_in_window','severity_high']].round(6)

# Sort by first_anomaly_high_ts asc then severity_high desc; null timestamps go last
agg['ts_sort'] = pd.to_datetime(agg['first_anomaly_high_ts'], utc=True).fillna(pd.Timestamp.max)
result = agg.sort_values(['ts_sort','severity_high'], ascending=[True, False]).head(30)

# Select requested columns and reset index
final_cols = ['cmdb_id','trace_name','window_count','anomaly_count_high','first_anomaly_high_ts','p95','max_value_in_window','severity_high']
result = result[final_cols].reset_index(drop=True)

result

```
Out[12]:
```
```python
summary = (
    "Summary of trace anomalies for candidate services (window 2022-03-21 21:30:00 → 21:59:00 UTC+8):\n\n"
    "- Scope: checked traces for adservice-1, checkoutservice-2, cartservice-2, cartservice-1, adservice2-0, "
    "currencyservice-0, frontend-2, frontend-1, frontend-0 that represent incoming calls from frontends or self metrics.\n\n"
    "- Timing: most matching trace series first crossed their global P95 at 2022-03-21 21:30:00 +0800 (earliest anomaly).\n\n"
    "- Highest relative severity (top examples):\n"
    "  * adservice-1 / trace.from_frontend-2.duration_p95 — severity ≈ 0.79 (p95 0.000323 → max 0.000578)\n"
    "  * adservice-1 / trace.from_frontend-2.duration_mean — severity ≈ 0.70 (p95 0.000081 → max 0.000138)\n"
    "  * cartservice-2 / trace.from_frontend-2.row_count — severity ≈ 0.11 (p95 22.55 → max 25.00)\n"
    "  * cartservice-1 / trace.from_frontend-2.row_count — severity ≈ 0.09 (p95 22.00 → max 24.00)\n"
    "  * cartservice-1 / trace.from_frontend-1.row_count — severity ≈ 0.06 (p95 17.00 → max 18.00)\n\n"
    "- Broad pattern: many incoming-from-frontend and trace.self series had anomaly_count_high > 0 (several with anomalies on multiple points). "
    "A number of trace.self.* and error_rate series have p95 == 0, so severity is undefined (NaN) even though they show high anomaly counts.\n\n"
    "Interpretation: at 21:30 many services observed elevated trace durations and increased row/traffic counts from frontends. "
    "The most significant relative increases are small-duration traces on adservice-1 (large relative jump from a very small baseline) and elevated row_count on cartservice and related services. "
    "This indicates both latency increases and higher incoming load from frontends starting at 21:30 +0800."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name  window_count  anomaly_count_high      first_anomaly_high_ts        p95  max_value_in_window  severity_high
0         adservice-1   trace.from_frontend-2.duration_p95            30                   2  2022-03-21 21:30:00 +0800   0.000323             0.000578       0.790305
1         adservice-1  trace.from_frontend-2.duration_mean            30                   2  2022-03-21 21:30:00 +0800   0.000081             0.000138       0.701603
2       cartservice-2      trace.from_frontend-2.row_count            30                   2  2022-03-21 21:30:00 +0800  22.550000            25.000000       0.108647
3       cartservice-1      trace.from_frontend-2.row_count            30                   4  2022-03-21 21:30:00 +0800  22.000000            24.000000       0.090909
4       cartservice-1      trace.from_frontend-1.row_count            30                   4  2022-03-21 21:30:00 +0800  17.000000            18.000000       0.058824
..                ...                                  ...           ...                 ...                        ...        ...                  ...            ...
25      cartservice-2              trace.self.duration_p95            30                  30  2022-03-21 21:30:00 +0800   0.000000             0.000000            NaN
26      cartservice-2                trace.self.error_rate            30                  30  2022-03-21 21:30:00 +0800   0.000000             0.000000            NaN
27  checkoutservice-2     trace.from_frontend-1.error_rate            19                  19  2022-03-21 21:30:00 +0800   0.000000             0.000000            NaN
28  checkoutservice-2     trace.from_frontend-2.error_rate            18                  18  2022-03-21 21:30:00 +0800   0.000000             0.000000            NaN
29  checkoutservice-2                trace.self.error_rate            26                  26  2022-03-21 21:30:00 +0800   0.000000             0.000000            NaN

[30 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[13]:

import pandas as pd
import numpy as np
import pytz

# Reuse previously loaded metric_df, grp (global P95), and any windowed variables if present; otherwise compute them
tz = pytz.timezone('Asia/Shanghai')

if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure global P95 per series exists (computed on full series BEFORE any window filtering)
if 'grp' not in globals():
    grp = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(p95=lambda x: x.quantile(0.95)).reset_index()

# Define incident window (we'll reuse the same window used previously)
start_local = pd.Timestamp('2022-03-21 21:30:00', tz=tz)
end_local   = pd.Timestamp('2022-03-21 21:59:00', tz=tz)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch   = int(end_local.tz_convert('UTC').timestamp())

# Windowed data with merged p95 (only for identifying anomalies in-window)
if 'window_with_p95' not in globals():
    window_df = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()
    window_with_p95 = window_df.merge(grp[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left', validate='m:1')
else:
    # ensure window_with_p95 corresponds to the same window; assume it does from prior steps
    window_df = window_with_p95[['timestamp','cmdb_id','kpi_name','value','p95']].copy()

# Drop series without global p95 (cannot evaluate)
window_with_p95 = window_with_p95.dropna(subset=['p95']).copy()

# Identify anomaly points (value >= global P95) and earliest anomaly per series within window
anomaly_points = window_with_p95[window_with_p95['value'] >= window_with_p95['p95']].copy()
first_anoms = anomaly_points.groupby(['cmdb_id','kpi_name'], as_index=False).agg(first_anomaly_ts_raw=('timestamp','min'))

# Precompute per-series max_value_in_window (needed for severity)
window_stats = window_with_p95.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    max_value_in_window=('value','max'),
    p95=('p95','first')
)

# Helper: format epoch -> Asia/Shanghai string
def fmt_ts_epoch(val):
    if pd.isna(val):
        return pd.NA
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %z')

# Target minutes to analyze
target_minutes_local = ['2022-03-21 21:45:00', '2022-03-21 21:58:00']
results = {}

for minute_str in target_minutes_local:
    target_local = pd.Timestamp(minute_str, tz=tz)
    target_epoch = int(target_local.tz_convert('UTC').timestamp())
    target_minute_epoch = (target_epoch // 60) * 60

    # Add minute_epoch for first_anoms if not present
    first_anoms['minute_epoch'] = (first_anoms['first_anomaly_ts_raw'] // 60) * 60

    # Filter series whose first_anomaly minute equals target minute
    first_min = first_anoms[first_anoms['minute_epoch'] == target_minute_epoch].copy()

    # Part 1: aggregate by cmdb_id -> top 10 by series_count
    if first_min.empty:
        cmdb_result = pd.DataFrame(columns=['cmdb_id','series_count','sample_kpi_examples'])
        series_result = pd.DataFrame(columns=['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity'])
    else:
        cmdb_agg = (
            first_min.groupby('cmdb_id', as_index=False)
            .agg(series_count=('kpi_name','size'),
                 sample_kpis=('kpi_name', lambda s: list(pd.unique(s)[:5])))
            .sort_values('series_count', ascending=False)
            .head(10)
            .reset_index(drop=True)
        )
        cmdb_agg['sample_kpi_examples'] = cmdb_agg['sample_kpis'].apply(lambda lst: ', '.join(lst))
        cmdb_result = cmdb_agg[['cmdb_id','series_count','sample_kpi_examples']]

        # Part 2: compute severity per series and return top 10 series by severity desc
        series_stats = first_min.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
        # severity: NaN if p95 == 0
        series_stats['severity'] = np.where(series_stats['p95']>0,
                                            (series_stats['max_value_in_window'] - series_stats['p95'])/series_stats['p95'],
                                            np.nan)
        series_stats['first_anomaly_ts'] = series_stats['first_anomaly_ts_raw'].apply(fmt_ts_epoch)
        # round numeric columns
        series_stats['p95'] = series_stats['p95'].round(6)
        series_stats['max_value_in_window'] = series_stats['max_value_in_window'].round(6)
        series_stats['severity'] = series_stats['severity'].round(6)

        series_result = series_stats.sort_values('severity', ascending=False).head(10)[
            ['cmdb_id','kpi_name','first_anomaly_ts','p95','max_value_in_window','severity']
        ].reset_index(drop=True)

    results[minute_str] = (cmdb_result, series_result)

# Display compact results for both minutes
results['2022-03-21 21:45:00'], results['2022-03-21 21:58:00']

```
Out[13]:
```
Summary for minute 2022-03-21 21:45:00 +0800

1) Top cmdb_id by number of series that first crossed P95 at 21:45 (top 10)
- adservice2 — 6 series (examples: runtime.jvm_threads_* metrics)
- checkoutservice-0 — 2 series (includes istio mesh source traces)
- cartservice2-0 — 1 series (container threads)
- frontend-2 — 1 series (istio/responses from frontend→checkoutservice)
- node-1 — 1 series (system.disk.used)
- node-2 — 1 series (system.disk.used)
- node-3 — 1 series (system.net.tcp.retrans_segs)
- node-5 — 1 series (system.io.svctm)
- productcatalogservice-0 — 1 series (container_network_receive_MB...)

2) Top 10 individual series (by severity) that first crossed at 21:45
- node-5 / system.io.svctm — p95=0.2655 → max=0.4000 — severity ≈ 0.5066
- adservice2 / runtime.jvm_threads_started — p95=0.6750 → max=0.916667 — severity ≈ 0.3580
- checkoutservice-0 / mesh.source.checkoutservice.emailservice.istio... — p95=0.003675 → max=0.004675 — severity ≈ 0.2721
- node-3 / system.net.tcp.retrans_segs — p95=0.2755 → max=0.3300 — severity ≈ 0.1978
- frontend-2 / mesh.source.frontend.checkoutservice.istio_res... — p95=1391 → max=1587.5 — severity ≈ 0.1413
- productcatalogservice-0 / container_network_receive_MB... — p95≈0.7676 → max≈0.8056 — severity ≈ 0.0495
- additional adservice2 runtime/threading metrics with smaller severities (~0.01–0.04)

Interpretation (21:45): a handful of node-level I/O/network metrics and JVM/threading metrics in adservice2 show the largest relative changes at this minute; some mesh/istio trace counts also increase.

---

Summary for minute 2022-03-21 21:58:00 +0800

1) Top cmdb_id by number of series that first crossed P95 at 21:58 (top 10)
- adservice — 19 series (various runtime/JVM metrics)
- adservice2 — 17 series
- cartservice-2 — 6 series
- cartservice-1 — 6 series
- cartservice-0 — 6 series
- cartservice2-0 — 6 series
- node-6 — 5 series (node-level metrics)
- currencyservice2-0 — 3 series
- checkoutservice-1 — 2 series
- currencyservice-0 — 2 series

2) Top 10 individual series (by severity) that first crossed at 21:58
- paymentservice2-0 / container.node-6.container_memory_mapped_file — p95≈559308.8 → max≈598016.0 — severity ≈ 0.0692
- node-6 / system.load.5 — p95=2.1365 → max=2.18 — severity ≈ 0.0204
- adservice / runtime.java_lang_Threading_CurrentThreadAllocatedBytes... — p95≈5.0339e9 → max≈5.0542e9 — severity ≈ 0.0040
- cartservice-0 / container_fs_usage_MB (/dev/vda1) — p95≈19.0087 → max≈19.0742 — severity ≈ 0.00345
- several adservice JVM memory/thread/gc metrics with small severities (~0.0027–0.0033)
- cartservice-2 / container_memory_max_usage_MB — small severity ≈ 0.0027

Interpretation (21:58): the 21:58 minute shows many JVM/memory and container filesystem metrics crossing P95 across adservice and cartservice families, and a notable memory-mapped-file increase on paymentservice2-0 and modest node-6 load rise. Severities here are much smaller than the peaks at 21:45 (largest ~0.07), indicating smaller relative deviations.

Overall takeaway:
- 21:45 is characterized by notable node I/O/network and threading spikes (higher relative severities).
- 21:58 shows broader but smaller-magnitude crossings across JVM/memory and container FS metrics, affecting many adservice/cartservice series.

The original code execution output of IPython Kernel is also provided below for reference:

((                   cmdb_id  series_count                                sample_kpi_examples
0               adservice2             6  runtime.java_lang_Threading_DaemonThreadCount,...
1        checkoutservice-0             2  mesh.source.checkoutservice.emailservice.istio...
2           cartservice2-0             1                 container.node-6.container_threads
3               frontend-2             1  mesh.source.frontend.checkoutservice.istio_res...
4                   node-1             1                                   system.disk.used
5                   node-2             1                                   system.disk.used
6                   node-3             1                        system.net.tcp.retrans_segs
7                   node-5             1                                    system.io.svctm
8  productcatalogservice-0             1  container.node-5.container_network_receive_MB....,                    cmdb_id                                           kpi_name           first_anomaly_ts          p95  max_value_in_window  severity
0                   node-5                                    system.io.svctm  2022-03-21 21:45:00 +0800     0.265500             0.400000  0.506591
1               adservice2                        runtime.jvm_threads_started  2022-03-21 21:45:00 +0800     0.675000             0.916667  0.358025
2        checkoutservice-0  mesh.source.checkoutservice.emailservice.istio...  2022-03-21 21:45:00 +0800     0.003675             0.004675  0.272109
3                   node-3                        system.net.tcp.retrans_segs  2022-03-21 21:45:00 +0800     0.275500             0.330000  0.197822
4               frontend-2  mesh.source.frontend.checkoutservice.istio_res...  2022-03-21 21:45:00 +0800  1391.000000          1587.500000  0.141265
5  productcatalogservice-0  container.node-5.container_network_receive_MB....  2022-03-21 21:45:00 +0800     0.767611             0.805634  0.049535
6               adservice2            runtime.jvm_threads_state.TIMED_WAITING  2022-03-21 21:45:00 +0800     5.675000             5.916667  0.042584
7               adservice2      runtime.java_lang_Threading_DaemonThreadCount  2022-03-21 21:45:00 +0800    15.675000            15.916667  0.015417
8               adservice2                         runtime.jvm_threads_daemon  2022-03-21 21:45:00 +0800    15.675000            15.916667  0.015417
9               adservice2            runtime.java_lang_Threading_ThreadCount  2022-03-21 21:45:00 +0800    17.675000            17.916667  0.013673), (              cmdb_id  series_count                                sample_kpi_examples
0           adservice            19  runtime.java_lang_Compilation_TotalCompilation...
1          adservice2            17  runtime.java_lang_GarbageCollector_CollectionC...
2       cartservice-2             6  container.node-5.container_fs_usage_MB./dev/vd...
3       cartservice-1             6  container.node-5.container_fs_usage_MB./dev/vd...
4       cartservice-0             6  container.node-5.container_fs_usage_MB./dev/vd...
5      cartservice2-0             6  container.node-6.container_fs_usage_MB./dev/vd...
6              node-6             5  system.disk.pct_usage, system.load.5, system.m...
7  currencyservice2-0             3  container.node-6.container_fs_usage_MB./dev/vd...
8   checkoutservice-1             2  container.node-5.container_fs_usage_MB./dev/vd...
9   currencyservice-0             2  container.node-5.container_fs_usage_MB./dev/vd...,              cmdb_id                                           kpi_name           first_anomaly_ts           p95  max_value_in_window  severity
0  paymentservice2-0      container.node-6.container_memory_mapped_file  2022-03-21 21:58:00 +0800  5.593088e+05         5.980160e+05  0.069205
1             node-6                                      system.load.5  2022-03-21 21:58:00 +0800  2.136500e+00         2.180000e+00  0.020360
2          adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  2022-03-21 21:58:00 +0800  5.033874e+09         5.054245e+09  0.004047
3      cartservice-0   container.node-5.container_fs_usage_MB./dev/vda1  2022-03-21 21:58:00 +0800  1.900869e+01         1.907422e+01  0.003447
4          adservice  runtime.java_lang_Threading_CurrentThreadUserTime  2022-03-21 21:58:00 +0800  6.467208e+09         6.488333e+09  0.003266
5          adservice  runtime.jvm_memory_pool_allocated_MB_total.Ede...  2022-03-21 21:58:00 +0800  3.246429e+04         3.256708e+04  0.003166
6          adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  2022-03-21 21:58:00 +0800  6.720741e+09         6.740778e+09  0.002981
7          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2022-03-21 21:58:00 +0800  3.020238e+03         3.028417e+03  0.002708
8          adservice  runtime.java_lang_GarbageCollector_CollectionC...  2022-03-21 21:58:00 +0800  3.020238e+03         3.028417e+03  0.002708
9      cartservice-2     container.node-5.container_memory_max_usage_MB  2022-03-21 21:58:00 +0800  7.964023e+01         7.985547e+01  0.002703))```
```